# 🔒 Security Audit & Hardening

This notebook audits your local environment for security issues and hardens your setup.
Run this **before** connecting any more real accounts.

## What's at Stake
Your project stores:
- **Plaid access tokens** → read access to your real brokerage accounts (holdings, balances, transactions)
- **Plaid API secret** → can create new connections to any institution on your account
- **Anthropic API key** → can make API calls billed to you
- **SendGrid/Slack/Pushover keys** → can send messages as you
- **Portfolio data** → your exact holdings, positions, and financial picture

None of this data enables someone to *move* money or execute trades, but it's still sensitive financial information.

## 1. File Permission Audit

In [1]:
import os
import stat
import json
import subprocess

PROJECT_ROOT = os.path.abspath("..")

SENSITIVE_FILES = [
    ".env",
    "access_tokens.json",
    "portfolio_snapshot.json",
    "enriched_portfolio.json",
]

SENSITIVE_DIRS = [
    "reports",
]

print("═" * 60)
print("  FILE PERMISSION AUDIT")
print("═" * 60)

issues = []

for filename in SENSITIVE_FILES:
    filepath = os.path.join(PROJECT_ROOT, filename)
    if not os.path.exists(filepath):
        print(f"  ⬜ {filename}: not found (OK if not yet created)")
        continue
    
    st = os.stat(filepath)
    mode = stat.S_IMODE(st.st_mode)
    mode_str = oct(mode)
    
    # Check if group or others can read
    if mode & (stat.S_IRGRP | stat.S_IROTH):
        print(f"  🔴 {filename}: {mode_str} — READABLE BY OTHERS")
        issues.append((filename, "Restrict permissions: chmod 600"))
    else:
        print(f"  🟢 {filename}: {mode_str} — owner-only ✓")

print()
if issues:
    print(f"  ⚠️  {len(issues)} issue(s) found. Run the fix cell below.")
else:
    print(f"  ✅ All sensitive files have proper permissions.")

════════════════════════════════════════════════════════════
  FILE PERMISSION AUDIT
════════════════════════════════════════════════════════════
  🟢 .env: 0o600 — owner-only ✓
  🟢 access_tokens.json: 0o600 — owner-only ✓
  🟢 portfolio_snapshot.json: 0o600 — owner-only ✓
  🔴 enriched_portfolio.json: 0o644 — READABLE BY OTHERS

  ⚠️  1 issue(s) found. Run the fix cell below.


In [2]:
# FIX: Lock down sensitive file permissions (owner read/write only)
print("Locking down file permissions...\n")

for filename in SENSITIVE_FILES:
    filepath = os.path.join(PROJECT_ROOT, filename)
    if os.path.exists(filepath):
        os.chmod(filepath, stat.S_IRUSR | stat.S_IWUSR)  # 600
        print(f"  🔒 {filename} → chmod 600 (owner read/write only)")

for dirname in SENSITIVE_DIRS:
    dirpath = os.path.join(PROJECT_ROOT, dirname)
    if os.path.exists(dirpath):
        os.chmod(dirpath, stat.S_IRWXU)  # 700
        print(f"  🔒 {dirname}/ → chmod 700 (owner only)")

print("\n✅ Permissions locked down.")

Locking down file permissions...

  🔒 .env → chmod 600 (owner read/write only)
  🔒 access_tokens.json → chmod 600 (owner read/write only)
  🔒 portfolio_snapshot.json → chmod 600 (owner read/write only)
  🔒 enriched_portfolio.json → chmod 600 (owner read/write only)
  🔒 reports/ → chmod 700 (owner only)

✅ Permissions locked down.


## 2. Git Leak Check

Verify that no secrets have been committed to git.

In [3]:
print("═" * 60)
print("  GIT HISTORY LEAK CHECK")
print("═" * 60)

issues = []

# Check .gitignore exists and has the right entries
gitignore_path = os.path.join(PROJECT_ROOT, ".gitignore")
if os.path.exists(gitignore_path):
    with open(gitignore_path) as f:
        gitignore_content = f.read()
    
    required_entries = [".env", "access_tokens.json", "reports/", "portfolio_snapshot.json", "enriched_portfolio.json"]
    for entry in required_entries:
        if entry in gitignore_content:
            print(f"  🟢 .gitignore contains '{entry}' ✓")
        else:
            print(f"  🔴 .gitignore MISSING '{entry}'")
            issues.append(f"Add '{entry}' to .gitignore")
else:
    print("  🔴 .gitignore not found!")
    issues.append("Create .gitignore")

print()

# Check if sensitive files are tracked by git
try:
    result = subprocess.run(
        ["git", "ls-files"],
        cwd=PROJECT_ROOT,
        capture_output=True,
        text=True,
    )
    tracked_files = result.stdout.strip().split("\n") if result.stdout.strip() else []
    
    danger_files = [".env", "access_tokens.json", "portfolio_snapshot.json", "enriched_portfolio.json"]
    for df in danger_files:
        if df in tracked_files:
            print(f"  🔴 '{df}' IS TRACKED BY GIT — secrets may be in commit history!")
            issues.append(f"Remove '{df}' from git: git rm --cached {df}")
        else:
            print(f"  🟢 '{df}' not tracked by git ✓")
except FileNotFoundError:
    print("  ⚠️  Git not found — skipping tracked file check.")

print()

# Search git log for accidental secret commits
try:
    result = subprocess.run(
        ["git", "log", "--all", "--oneline", "--diff-filter=A", "--name-only"],
        cwd=PROJECT_ROOT,
        capture_output=True,
        text=True,
    )
    log_output = result.stdout
    
    for danger in [".env", "access_tokens.json"]:
        if danger in log_output:
            print(f"  🔴 '{danger}' appears in git history — may have been committed before!")
            issues.append(f"'{danger}' found in git history — rotate all secrets in that file")
        else:
            print(f"  🟢 '{danger}' never appears in git history ✓")
except FileNotFoundError:
    pass

print()
if issues:
    print(f"⚠️  {len(issues)} issue(s):")
    for issue in issues:
        print(f"  → {issue}")
else:
    print("✅ No git leak issues detected.")

════════════════════════════════════════════════════════════
  GIT HISTORY LEAK CHECK
════════════════════════════════════════════════════════════
  🟢 .gitignore contains '.env' ✓
  🟢 .gitignore contains 'access_tokens.json' ✓
  🟢 .gitignore contains 'reports/' ✓
  🟢 .gitignore contains 'portfolio_snapshot.json' ✓
  🟢 .gitignore contains 'enriched_portfolio.json' ✓

  🟢 '.env' not tracked by git ✓
  🟢 'access_tokens.json' not tracked by git ✓
  🟢 'portfolio_snapshot.json' not tracked by git ✓
  🟢 'enriched_portfolio.json' not tracked by git ✓

  🟢 '.env' never appears in git history ✓
  🟢 'access_tokens.json' never appears in git history ✓

✅ No git leak issues detected.


## 3. Encrypt Access Tokens at Rest

Instead of storing access tokens in plaintext JSON, we encrypt them using a password-derived key.
This means even if someone accesses your filesystem, the tokens are unreadable without your password.

We use **Fernet symmetric encryption** (AES-128-CBC) from the `cryptography` library.

In [4]:
# First, check if cryptography is installed
try:
    from cryptography.fernet import Fernet
    from cryptography.hazmat.primitives import hashes
    from cryptography.hazmat.primitives.kdf.pbkdf2 import PBKDF2HMAC
    import base64
    print("✅ cryptography library available")
except ImportError:
    print("⚠️  cryptography not installed. Run:")
    print("   uv add cryptography")
    print("   Then re-run this cell.")

✅ cryptography library available


In [5]:
import getpass
import base64
from cryptography.fernet import Fernet
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.kdf.pbkdf2 import PBKDF2HMAC

TOKENS_FILE = os.path.join(PROJECT_ROOT, "access_tokens.json")
ENCRYPTED_FILE = os.path.join(PROJECT_ROOT, "access_tokens.enc")
SALT_FILE = os.path.join(PROJECT_ROOT, ".token_salt")


def derive_key(password: str, salt: bytes) -> bytes:
    """Derive a Fernet key from a password using PBKDF2."""
    kdf = PBKDF2HMAC(
        algorithm=hashes.SHA256(),
        length=32,
        salt=salt,
        iterations=480_000,  # OWASP recommended minimum
    )
    return base64.urlsafe_b64encode(kdf.derive(password.encode()))


def encrypt_tokens(password: str):
    """Encrypt access_tokens.json → access_tokens.enc"""
    if not os.path.exists(TOKENS_FILE):
        print("⚠️  No access_tokens.json found to encrypt.")
        return
    
    # Generate or load salt
    if os.path.exists(SALT_FILE):
        with open(SALT_FILE, "rb") as f:
            salt = f.read()
    else:
        salt = os.urandom(16)
        with open(SALT_FILE, "wb") as f:
            f.write(salt)
        os.chmod(SALT_FILE, stat.S_IRUSR | stat.S_IWUSR)
    
    key = derive_key(password, salt)
    fernet = Fernet(key)
    
    with open(TOKENS_FILE, "r") as f:
        plaintext = f.read().encode()
    
    encrypted = fernet.encrypt(plaintext)
    
    with open(ENCRYPTED_FILE, "wb") as f:
        f.write(encrypted)
    os.chmod(ENCRYPTED_FILE, stat.S_IRUSR | stat.S_IWUSR)
    
    print(f"✅ Encrypted tokens saved to {ENCRYPTED_FILE}")
    print(f"   You can now delete the plaintext file:")
    print(f"   rm {TOKENS_FILE}")


def decrypt_tokens(password: str) -> dict:
    """Decrypt access_tokens.enc → dict"""
    if not os.path.exists(ENCRYPTED_FILE):
        raise FileNotFoundError("No encrypted token file found.")
    if not os.path.exists(SALT_FILE):
        raise FileNotFoundError("No salt file found.")
    
    with open(SALT_FILE, "rb") as f:
        salt = f.read()
    
    key = derive_key(password, salt)
    fernet = Fernet(key)
    
    with open(ENCRYPTED_FILE, "rb") as f:
        encrypted = f.read()
    
    decrypted = fernet.decrypt(encrypted)
    return json.loads(decrypted.decode())


print("🔐 Token encryption utilities loaded.")
print("   Run the next cell to encrypt your tokens.")

🔐 Token encryption utilities loaded.
   Run the next cell to encrypt your tokens.


In [6]:
# Encrypt your tokens (interactive — will prompt for password)
if os.path.exists(TOKENS_FILE):
    print("Enter a password to encrypt your access tokens.")
    print("You'll need this password every time the pipeline runs.\n")
    
    password = getpass.getpass("Encryption password: ")
    confirm = getpass.getpass("Confirm password: ")
    
    if password != confirm:
        print("❌ Passwords don't match. Try again.")
    elif len(password) < 8:
        print("❌ Password must be at least 8 characters.")
    else:
        encrypt_tokens(password)
        
        # Verify by decrypting
        try:
            tokens = decrypt_tokens(password)
            print(f"\n✅ Verification: successfully decrypted {len(tokens)} tokens")
            print(f"\n🗑️  To complete hardening, delete the plaintext file:")
            print(f"   rm {TOKENS_FILE}")
            print(f"\n   Your notebooks will need to be updated to use decrypt_tokens().")
            print(f"   See the helper module created in the next cell.")
        except Exception as e:
            print(f"❌ Verification failed: {e}")
else:
    print("⚠️  No access_tokens.json found.")
    if os.path.exists(ENCRYPTED_FILE):
        print("   But access_tokens.enc exists — tokens are already encrypted ✅")

Enter a password to encrypt your access tokens.
You'll need this password every time the pipeline runs.



Encryption password:  ········
Confirm password:  ········


✅ Encrypted tokens saved to /Users/sadmin/projects/fin_analyst/financial-agent 3/access_tokens.enc
   You can now delete the plaintext file:
   rm /Users/sadmin/projects/fin_analyst/financial-agent 3/access_tokens.json

✅ Verification: successfully decrypted 1 tokens

🗑️  To complete hardening, delete the plaintext file:
   rm /Users/sadmin/projects/fin_analyst/financial-agent 3/access_tokens.json

   Your notebooks will need to be updated to use decrypt_tokens().
   See the helper module created in the next cell.


## 4. Create Secure Token Loader Module

This creates a helper module that notebooks can import.
It tries encrypted tokens first, falls back to plaintext.

In [7]:
secure_loader_code = '''"""Secure token loader — tries encrypted file first, falls back to plaintext."""

import json
import os
import getpass
import stat

TOKENS_FILE = os.path.join(os.path.dirname(__file__), "access_tokens.json")
ENCRYPTED_FILE = os.path.join(os.path.dirname(__file__), "access_tokens.enc")
SALT_FILE = os.path.join(os.path.dirname(__file__), ".token_salt")


def load_tokens(password: str = None) -> dict:
    """Load access tokens, preferring encrypted file.
    
    If encrypted file exists, prompts for password (or uses provided one).
    Falls back to plaintext JSON if no encrypted file.
    """
    # Try encrypted first
    if os.path.exists(ENCRYPTED_FILE) and os.path.exists(SALT_FILE):
        try:
            from cryptography.fernet import Fernet
            from cryptography.hazmat.primitives import hashes
            from cryptography.hazmat.primitives.kdf.pbkdf2 import PBKDF2HMAC
            import base64
            
            if password is None:
                password = getpass.getpass("Token decryption password: ")
            
            with open(SALT_FILE, "rb") as f:
                salt = f.read()
            
            kdf = PBKDF2HMAC(
                algorithm=hashes.SHA256(),
                length=32,
                salt=salt,
                iterations=480_000,
            )
            key = base64.urlsafe_b64encode(kdf.derive(password.encode()))
            fernet = Fernet(key)
            
            with open(ENCRYPTED_FILE, "rb") as f:
                encrypted = f.read()
            
            decrypted = fernet.decrypt(encrypted)
            return json.loads(decrypted.decode())
        except Exception as e:
            raise ValueError(f"Failed to decrypt tokens: {e}")
    
    # Fall back to plaintext
    if os.path.exists(TOKENS_FILE):
        with open(TOKENS_FILE, "r") as f:
            return json.load(f)
    
    raise FileNotFoundError(
        "No access tokens found. Run notebook 01 to connect accounts."
    )


def save_tokens(tokens: dict):
    """Save tokens to plaintext file with restricted permissions."""
    with open(TOKENS_FILE, "w") as f:
        json.dump(tokens, f, indent=2)
    os.chmod(TOKENS_FILE, stat.S_IRUSR | stat.S_IWUSR)
'''

loader_path = os.path.join(PROJECT_ROOT, "token_store.py")
with open(loader_path, "w") as f:
    f.write(secure_loader_code)

print(f"✅ Created {loader_path}")
print(f"")
print(f"   Usage in notebooks:")
print(f"   from token_store import load_tokens")
print(f"   access_tokens = load_tokens()")
print(f"")
print(f"   This replaces the json.load(open('access_tokens.json')) pattern.")
print(f"   It auto-detects encrypted vs plaintext tokens.")

✅ Created /Users/sadmin/projects/fin_analyst/financial-agent 3/token_store.py

   Usage in notebooks:
   from token_store import load_tokens
   access_tokens = load_tokens()

   This replaces the json.load(open('access_tokens.json')) pattern.
   It auto-detects encrypted vs plaintext tokens.


## 5. Environment Variable Audit

In [8]:
from dotenv import dotenv_values

print("═" * 60)
print("  ENVIRONMENT VARIABLE AUDIT")
print("═" * 60)

env_path = os.path.join(PROJECT_ROOT, ".env")

if not os.path.exists(env_path):
    print("  ⚠️  No .env file found. Create one from .env.example.")
else:
    env_vars = dotenv_values(env_path)
    
    # Check each key is set (don't print values!)
    critical_keys = [
        ("PLAID_CLIENT_ID", True),
        ("PLAID_SECRET", True),
        ("PLAID_ENV", True),
        ("ANTHROPIC_API_KEY", True),
        ("SENDGRID_API_KEY", False),
        ("NOTIFICATION_EMAIL", False),
        ("SLACK_WEBHOOK_URL", False),
        ("PUSHOVER_USER_KEY", False),
    ]
    
    for key, required in critical_keys:
        value = env_vars.get(key, "")
        if value and value not in ["your_client_id_here", "your_sandbox_secret_here", "your_anthropic_api_key_here"]:
            # Show only first/last 4 chars
            masked = value[:4] + "*" * max(0, len(value) - 8) + value[-4:] if len(value) > 12 else "****"
            print(f"  🟢 {key}: {masked}")
        elif required:
            print(f"  🔴 {key}: NOT SET (required)")
        else:
            print(f"  ⬜ {key}: not set (optional)")
    
    # Check env isn't world-readable
    env_mode = stat.S_IMODE(os.stat(env_path).st_mode)
    print(f"\n  File permissions: {oct(env_mode)}")
    if env_mode & (stat.S_IRGRP | stat.S_IROTH):
        print(f"  🔴 .env is readable by other users! Run: chmod 600 .env")
    else:
        print(f"  🟢 .env is owner-only ✓")

════════════════════════════════════════════════════════════
  ENVIRONMENT VARIABLE AUDIT
════════════════════════════════════════════════════════════
  🟢 PLAID_CLIENT_ID: 6993****************d353
  🟢 PLAID_SECRET: 3cb3**********************2be7
  🟢 PLAID_ENV: ****
  🟢 ANTHROPIC_API_KEY: sk-a****************************************************************************************************PgAA
  🟢 SENDGRID_API_KEY: SG.c*************************************************************Z_3o
  🟢 NOTIFICATION_EMAIL: 94.m***********.com
  ⬜ SLACK_WEBHOOK_URL: not set (optional)
  ⬜ PUSHOVER_USER_KEY: not set (optional)

  File permissions: 0o600
  🟢 .env is owner-only ✓


## 6. Network Security Check

In [9]:
print("═" * 60)
print("  NETWORK & SYSTEM SECURITY")
print("═" * 60)

import platform
import shutil

# Check OS
os_name = platform.system()
print(f"\n  OS: {platform.platform()}")

# Check disk encryption
if os_name == "Darwin":  # macOS
    try:
        result = subprocess.run(
            ["fdesetup", "status"],
            capture_output=True, text=True,
        )
        if "On" in result.stdout:
            print(f"  🟢 FileVault disk encryption: ENABLED ✓")
        else:
            print(f"  🔴 FileVault disk encryption: DISABLED")
            print(f"     → Enable at: System Settings → Privacy & Security → FileVault")
            print(f"     → This encrypts your entire disk, protecting tokens if laptop is stolen")
    except Exception:
        print(f"  ⚠️  Could not check FileVault status")
    
    # Check firewall
    try:
        result = subprocess.run(
            ["/usr/libexec/ApplicationFirewall/socketfilterfw", "--getglobalstate"],
            capture_output=True, text=True,
        )
        if "enabled" in result.stdout.lower():
            print(f"  🟢 macOS Firewall: ENABLED ✓")
        else:
            print(f"  🟡 macOS Firewall: DISABLED")
            print(f"     → Enable at: System Settings → Network → Firewall")
    except Exception:
        print(f"  ⚠️  Could not check firewall status")

elif os_name == "Linux":
    # Check for LUKS encryption
    try:
        result = subprocess.run(["lsblk", "-o", "NAME,TYPE"], capture_output=True, text=True)
        if "crypt" in result.stdout:
            print(f"  🟢 Disk encryption: LUKS detected ✓")
        else:
            print(f"  🟡 Disk encryption: No LUKS volumes detected")
    except Exception:
        pass

elif os_name == "Windows":
    print(f"  ℹ️  Check BitLocker: Settings → Privacy & security → Device encryption")

# Check if any server is running on common ports
print(f"\n  Checking for open local servers...")
try:
    import socket
    for port in [5000, 5555, 8888, 8080]:
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        sock.settimeout(0.5)
        result = sock.connect_ex(('127.0.0.1', port))
        sock.close()
        if result == 0:
            print(f"  🟡 Port {port}: OPEN (check what's running — only needed during Link flow)")
        else:
            print(f"  🟢 Port {port}: closed ✓")
except Exception:
    pass

════════════════════════════════════════════════════════════
  NETWORK & SYSTEM SECURITY
════════════════════════════════════════════════════════════

  OS: macOS-26.2-x86_64-i386-64bit
  🟢 FileVault disk encryption: ENABLED ✓
  🟡 macOS Firewall: DISABLED
     → Enable at: System Settings → Network → Firewall

  Checking for open local servers...
  🟡 Port 5000: OPEN (check what's running — only needed during Link flow)
  🟡 Port 5555: OPEN (check what's running — only needed during Link flow)
  🟡 Port 8888: OPEN (check what's running — only needed during Link flow)
  🟢 Port 8080: closed ✓


## 7. Security Summary & Checklist

In [10]:
print("""
═══════════════════════════════════════════════════════════
  SECURITY HARDENING CHECKLIST
═══════════════════════════════════════════════════════════

Files & Permissions:
  □ .env has chmod 600 (owner-only)
  □ access_tokens.json has chmod 600 (or encrypted)
  □ reports/ directory has chmod 700
  □ All sensitive files are in .gitignore
  □ No secrets in git history (check with audit above)

Encryption:
  □ FileVault / BitLocker / LUKS enabled (full disk encryption)
  □ Access tokens encrypted with token_store.py (optional but recommended)

Git & GitHub:
  □ Repo is PRIVATE on GitHub
  □ No .env or access_tokens.json in any commit
  □ If secrets were ever committed: rotate ALL affected keys

API Key Hygiene:
  □ Plaid production secret ≠ sandbox secret
  □ Anthropic API key has spend limits set (console.anthropic.com)
  □ SendGrid API key scoped to Mail Send only (not Full Access)

Network:
  □ connect_real_account.py server only runs when actively connecting
  □ macOS Firewall enabled
  □ No unnecessary servers running on local ports

Plaid-Specific:
  □ Monitor Plaid billing dashboard for unexpected items
  □ Delete Items you no longer need (/item/remove) to stop billing
  □ Plaid access tokens are READ-ONLY (cannot move money)
  □ If a token is compromised: call /item/remove immediately,
    then change your brokerage password

Ongoing:
  □ Keep dependencies updated: uv sync --upgrade
  □ Rotate Plaid secret periodically (dashboard → Developers → Keys)
  □ Review Plaid billing monthly
  □ Don't run notebooks on public/shared machines

═══════════════════════════════════════════════════════════
""")


═══════════════════════════════════════════════════════════
  SECURITY HARDENING CHECKLIST
═══════════════════════════════════════════════════════════

Files & Permissions:
  □ .env has chmod 600 (owner-only)
  □ access_tokens.json has chmod 600 (or encrypted)
  □ reports/ directory has chmod 700
  □ All sensitive files are in .gitignore
  □ No secrets in git history (check with audit above)

Encryption:
  □ FileVault / BitLocker / LUKS enabled (full disk encryption)
  □ Access tokens encrypted with token_store.py (optional but recommended)

Git & GitHub:
  □ Repo is PRIVATE on GitHub
  □ No .env or access_tokens.json in any commit
  □ If secrets were ever committed: rotate ALL affected keys

API Key Hygiene:
  □ Plaid production secret ≠ sandbox secret
  □ Anthropic API key has spend limits set (console.anthropic.com)
  □ SendGrid API key scoped to Mail Send only (not Full Access)

Network:
  □ connect_real_account.py server only runs when actively connecting
  □ macOS Firewall enabl

## What CAN'T Be Stolen Even If Tokens Leak

It's important to understand the **blast radius** if something goes wrong:

| If this leaks... | An attacker CAN... | An attacker CANNOT... |
|---|---|---|
| Plaid access token | See your holdings, balances, transactions | Move money, execute trades, change passwords |
| Plaid API secret | Create new Link tokens (but still needs user auth) | Access existing accounts without tokens |
| Anthropic API key | Make API calls on your bill | Access any financial data |
| .env file | All of the above | Execute trades or move money |

**Bottom line:** A leak exposes your financial *information* but cannot cause financial *loss*. Still worth protecting seriously — identity theft and targeted phishing use exactly this kind of data.